# Data Cleaning 

## Bước 1 Xử lý những Giá trị thiếu (Missing values) và xóa những cột không quan trọng

In [1]:
import pandas as pd 
import numpy as np
# 1. Đọc dữ liệu từ file gốc
df = pd.read_csv('Bangladesh_database_Final_Merged.csv')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4608 entries, 0 to 4607
Data columns (total 61 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Area                    4608 non-null   int64  
 1   AP Ratio                4608 non-null   str    
 2   District                4608 non-null   str    
 3   Season                  4608 non-null   str    
 4   Avg Temp                4608 non-null   float64
 5   Avg Humidity            4608 non-null   float64
 6   Crop Name               4608 non-null   str    
 7   Transplant              4608 non-null   str    
 8   Growth                  4608 non-null   str    
 9   Harvest                 4608 non-null   str    
 10  Production              4608 non-null   int64  
 11  Max Temp                4608 non-null   int64  
 12  Min Temp                4608 non-null   float64
 13  Max Relative Humidity   4608 non-null   int64  
 14  Min Relative Humidity   4608 non-null   int64  
 15

In [2]:
df

,Area,AP Ratio,District,Season,Avg Temp,Avg Humidity,Crop Name,Transplant,Growth,Harvest,...,Temp_Min,Heat_Stress_Days,Wind_Mean,Wind_Max,Wind_Min,Temp_Range,Wind_Range,Rain_Temp_Ratio,Extreme_Heat_Risk,Is_Extreme_Heat
0,259,0.7722007722,Bagerhat,Rabi,17.75,55.0,Wheat,Dec,Jan to March,April,...,8.35,0,1.452917,4.33,0.04,22.23,4.29,11.23,Low Risk,0
1,421,2.247030879,Bagerhat,Rabi,20.50,60.0,Maize 2,Dec,Jan to March,April,...,8.35,0,1.452917,4.33,0.04,22.23,4.29,11.23,Low Risk,0
2,145439,1.847881242,Bagerhat,Rabi,26.00,72.5,Boro,Nov,Dec to March,April,...,8.35,0,1.452917,4.33,0.04,22.23,4.29,11.23,Low Risk,0
3,627,3.50877193,Bagerhat,Rabi,20.50,80.0,Sweet Potato,Nov,Dec to March,April,...,8.35,0,1.452917,4.33,0.04,22.23,4.29,11.23,Low Risk,0
4,383,4.994778068,Bagerhat,Kharif 1,31.50,70.0,Mango,April,No need to do,April to June,...,16.01,54,2.481885,5.86,0.04,25.25,5.82,22.73,High Risk,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4603,381,1.818897638,Thakurgaon,Rabi,25.00,80.0,Danta,March,April to Aug,Sep,...,8.94,0,1.245417,3.90,0.06,21.45,3.84,4.19,Low Risk,0
4604,935,3.068449198,Thakurgaon,Kharif 1,11.50,77.5,Mukhi Kachu,April,May to Sep,Sep to Feb,...,15.50,46,2.537541,6.30,0.09,25.43,6.21,36.58,High Risk,1
4605,58,1.189655172,Thakurgaon,Kharif 1,27.50,90.0,Kolmi Shak,March,April to Aug,Sep to Oct,...,15.50,46,2.537541,6.30,0.09,25.43,6.21,36.58,High Risk,1
4606,146,7.575342466,Thakurgaon,Kharif 1,29.00,67.5,Guava,April,No need to do,Throuout The year,...,15.50,46,2.537541,6.30,0.09,25.43,6.21,36.58,High Risk,1


#### Xem những bị thiếu những giá trị nào 

In [3]:
pd.set_option('display.max_rows', None)

df.isnull().sum()


Area                      0
AP Ratio                  0
District                  0
Season                    0
Avg Temp                  0
Avg Humidity              0
Crop Name                 0
Transplant                0
Growth                    0
Harvest                   0
Production                0
Max Temp                  0
Min Temp                  0
Max Relative Humidity     0
Min Relative Humidity     0
Year                      0
EVI                       0
LAI                       0
FPAR                      0
LST_Kelvin                0
Soil_Moisture_mm          0
NDVI_Season_Mean          0
NDVI_Season_Max           0
NDVI_Season_Min           0
NDVI_Season_Std           0
NDVI_Season_Range         0
NDVI_Season_CV            0
Avg_Salinity_Index        0
pH                        0
Organic_Carbon            0
Nitrogen                  0
Clay                      0
Sand                      0
Silt                      0
Bulk_Density              0
valid_points        

#### Xóa những cột không liên quan 

In [4]:
# Danh sách các cột đề xuất xóa
cols_to_drop = [
    # 1. Dữ liệu định danh/vị trí không đóng góp vào mô hình chung
    'grid_log',        # Thường là mã định danh lưới địa lý
    'valid_points',    # Chỉ số kỹ thuật khi thu thập dữ liệu
    'Latitude',        # Đã có cột 'District' đại diện cho khu vực
    'Longitude',       # Đã có cột 'District' đại diện cho khu vực
    
    # 2. Cột trùng lặp thông tin hoặc tính toán trực tiếp từ cột khác
    # Các cột Temp_Mean, Temp_Max... thường trùng lặp với Avg Temp, Max Temp
    'Temp_Mean',       # Trùng lặp với Avg Temp
    'Temp_Max',        # Trùng lặp với Max Temp
    'Temp_Min',        # Trùng lặp với Min Temp
    'Temp_Range',      # Có thể tính bằng Max - Min
    'Wind_Range'      # Có thể tính bằng Max - Min
    
    # 3. Cột chứa thông tin về Target (nếu bạn dự báo Production thì xóa AP Ratio và ngược lại)
    # Ở đây chúng ta giữ AP Ratio làm target chính, nên có thể cân nhắc xóa:
    
]

# Thực hiện xóa
df = df.drop(columns=cols_to_drop)

print(f"Số lượng cột còn lại: {df.shape[1]}")
print("Các cột đã giữ lại bao gồm các chỉ số quan trọng về Soil, NDVI, Weather và Crop Info.")

Số lượng cột còn lại: 52
Các cột đã giữ lại bao gồm các chỉ số quan trọng về Soil, NDVI, Weather và Crop Info.


## Bước 2 :Xử lý những giá trị trùng lặp 

In [5]:
df = df.drop_duplicates()

## Bước 3 : Xử lý lỗi sai định dạng 

In [6]:


# Giả sử bạn tiếp tục với df đã được xóa dòng lỗi ở bước trước
# df = pd.read_csv('Bangladesh_database_Cleaned_Step1.csv')

# 1. Danh sách các cột phân loại cần chuẩn hóa định dạng chuỗi
categorical_cols = [
    'District', 'Season', 'Crop Name', 'Transplant', 'Growth', 'Harvest', 
    'pH_Suitability', 'Compaction_Risk', 'Dominant_Soil_Texture', 
    'Water_Availability_Cat', 'Extreme_Heat_Risk'
]

for col in categorical_cols:
    if col in df.columns:
        # Xóa khoảng trắng thừa ở đầu/cuối và chuyển về chữ in hoa đầu từ (Capitalize)
        # Việc này giúp tránh trường hợp 'Jashore' và 'Jashore ' bị coi là 2 quận khác nhau
        df[col] = df[col].astype(str).str.strip().str.title()

# 2. Xử lý riêng các cột chu kỳ mùa vụ (thường bị sai định dạng dải tháng)
# Ví dụ: "April to Aug" vs "April-Aug"
date_cols = ['Transplant', 'Growth', 'Harvest']
for col in date_cols:
    if col in df.columns:
        df[col] = df[col].str.replace(' To ', ' to ', case=False)
        df[col] = df[col].str.replace('-', ' to ')

# 3. Ép kiểu dữ liệu số cho các cột tọa độ và chỉ số môi trường 
# (Đảm bảo không còn định dạng 'object' do dữ liệu nhiễu)
numeric_fix_cols = [
    'Avg Temp', 'Avg Humidity', 'Max Temp', 'Min Temp', 
    'Rainfall', 'Wind_Mean', 'pH'
]

for col in numeric_fix_cols:
    if col in df.columns:
        # Chuyển đổi sang số, các giá trị lỗi định dạng không chuyển được sẽ thành NaN
        df[col] = pd.to_numeric(df[col], errors='coerce')

# 4. Xử lý giá trị NaN sau khi ép kiểu (nếu có)
# Điền NaN bằng giá trị trung bình của cột đó để tránh lỗi mô hình
for col in numeric_fix_cols:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].mean())

# 5. Kiểm tra kết quả
print("Đã chuẩn hóa định dạng cho các cột phân loại và số.")
print(df[categorical_cols].head())

Đã chuẩn hóa định dạng cho các cột phân loại và số.
   District    Season     Crop Name Transplant          Growth        Harvest  \
0  Bagerhat      Rabi         Wheat        Dec  Jan  to  March          April   
1  Bagerhat      Rabi       Maize 2        Dec  Jan  to  March          April   
2  Bagerhat      Rabi          Boro        Nov    Dec to March          April   
3  Bagerhat      Rabi  Sweet Potato        Nov    Dec to March          April   
4  Bagerhat  Kharif 1         Mango      April   No Need to Do  April to June   

     pH_Suitability Compaction_Risk Dominant_Soil_Texture  \
0  Optimal (Tối Ưu)             Low   Loamy (Thịt/Phù Sa)   
1  Optimal (Tối Ưu)             Low   Loamy (Thịt/Phù Sa)   
2  Optimal (Tối Ưu)             Low   Loamy (Thịt/Phù Sa)   
3  Optimal (Tối Ưu)             Low   Loamy (Thịt/Phù Sa)   
4  Optimal (Tối Ưu)             Low   Loamy (Thịt/Phù Sa)   

  Water_Availability_Cat Extreme_Heat_Risk  
0                Optimal          Low Risk  
1   

In [7]:
df.head()

,Area,AP Ratio,District,Season,Avg Temp,Avg Humidity,Crop Name,Transplant,Growth,Harvest,...,Moisture_Ratio,Water_Availability_Cat,Rainfall,Heat_Stress_Days,Wind_Mean,Wind_Max,Wind_Min,Rain_Temp_Ratio,Extreme_Heat_Risk,Is_Extreme_Heat
0,259,0.7722007722,Bagerhat,Rabi,17.75,55.0,Wheat,Dec,Jan to March,April,...,1.4035,Optimal,220.43,0,1.452917,4.33,0.04,11.23,Low Risk,0
1,421,2.247030879,Bagerhat,Rabi,20.50,60.0,Maize 2,Dec,Jan to March,April,...,1.4035,Optimal,220.43,0,1.452917,4.33,0.04,11.23,Low Risk,0
2,145439,1.847881242,Bagerhat,Rabi,26.00,72.5,Boro,Nov,Dec to March,April,...,1.3611,Optimal,220.43,0,1.452917,4.33,0.04,11.23,Low Risk,0
3,627,3.50877193,Bagerhat,Rabi,20.50,80.0,Sweet Potato,Nov,Dec to March,April,...,1.3611,Optimal,220.43,0,1.452917,4.33,0.04,11.23,Low Risk,0
4,383,4.994778068,Bagerhat,Kharif 1,31.50,70.0,Mango,April,No Need to Do,April to June,...,1.3033,Optimal,656.33,54,2.481885,5.86,0.04,22.73,High Risk,1


## Bước 4: Xử lý những giá trị sai hoặc bất hợp lý (Invaild/incorrect data) 

#### 2. Xử lý lỗi #DIV/0! trong cột 'AP Ratio'
Thay thế chuỗi lỗi bằng NaN và chuyển sang kiểu số (float)
Xóa những cột target lỗi 

In [8]:
initial_count = len(df)
df['AP Ratio'] = df['AP Ratio'].replace('#DIV/0!', np.nan)
df['AP Ratio'] = pd.to_numeric(df['AP Ratio'], errors='coerce')
condition_to_drop = (df['Production'] <= 0) | (df['Area'] <= 0) | (df['AP Ratio'].isna())
df = df.drop(df[condition_to_drop].index)
final_count = len(df)
print(f"Số lượng dòng ban đầu: {initial_count}")
print(f"Số lượng dòng đã xóa: {initial_count - final_count}")
print(f"Số lượng dòng còn lại sau khi làm sạch: {final_count}")

Số lượng dòng ban đầu: 4608
Số lượng dòng đã xóa: 429
Số lượng dòng còn lại sau khi làm sạch: 4179


#### Xem Dữ liệu crop name có bao nhiêu loại cây 

In [9]:
df.columns = df.columns.str.strip()

crop_count = df['Crop Name'].value_counts()
print(crop_count)

Crop Name
Boro              64
Sweet Potato      64
Mango             64
Jack Fruit        64
Ripe Papaya       64
Aus               64
Karala            64
Sugarcane         64
Green Papaya      64
Amra              64
Wood Apple        64
Green Coconut     64
Barbati           64
Puishak           64
Palong Shak       64
Garlic            64
Cauliflower       64
Cabbage           64
Boroi             64
Banana            64
Lemon             64
Gourd             64
Betelnut          64
Chalkumra         64
Black Berry       64
Jhinga            64
Lady’S Finger     64
Radish            64
Chili             64
Onion             64
Cucumber          64
Jambura           64
Beans             64
Aman              64
Pumpkin           64
Lal Shak          64
Laushak           64
Danta Shak        64
Danta             64
Guava             64
Rape & Mustard    63
Taramind          63
Kakrol            63
Maize 2           62
Lentil            62
Green Palmyra     62
Sesame            61
Dat

### Xóa giá trị lỗi trong crop name

In [10]:
df = df[df['Crop Name'].str.strip() != '#Ref!']
df['Crop Name'].value_counts()

Crop Name
Boro              64
Sweet Potato      64
Mango             64
Jack Fruit        64
Ripe Papaya       64
Aus               64
Karala            64
Sugarcane         64
Green Papaya      64
Amra              64
Wood Apple        64
Green Coconut     64
Barbati           64
Puishak           64
Palong Shak       64
Garlic            64
Cauliflower       64
Cabbage           64
Boroi             64
Banana            64
Lemon             64
Gourd             64
Betelnut          64
Chalkumra         64
Black Berry       64
Jhinga            64
Lady’S Finger     64
Radish            64
Chili             64
Onion             64
Cucumber          64
Jambura           64
Beans             64
Aman              64
Pumpkin           64
Lal Shak          64
Laushak           64
Danta Shak        64
Danta             64
Guava             64
Rape & Mustard    63
Taramind          63
Kakrol            63
Maize 2           62
Lentil            62
Green Palmyra     62
Sesame            61
Dat

In [11]:
str_cols = df.select_dtypes(include=['object']).columns

for col in str_cols:
    print(f"\nColumn: {col}")
    print(df[col].unique())


Column: District
<StringArray>
[        'Bagerhat',        'Bandarban',          'Barguna',
         'Barishal',            'Bhola',           'Bogura',
     'Brahmanbaria',         'Chandpur', 'Chapai Nawabganj',
       'Chattogram',        'Chuadanga',        'Coxsbazar',
          'Cumilla',            'Dhaka',         'Dinajpur',
         'Faridpur',             'Feni',        'Gaibandha',
          'Gazipur',        'Gopalganj',         'Habiganj',
         'Jamalpur',          'Jashore',       'Jhallokati',
        'Jhenaidah',        'Joypurhat',      'Khagrachari',
           'Khulna',      'Kishoreganj',         'Kurigram',
          'Kushtia',       'Lakshmipur',      'Lalmonirhat',
        'Madaripur',           'Magura',        'Manikganj',
         'Meherpur',      'Moulvibazar',       'Munshiganj',
       'Mymensingh',          'Naogaon',           'Narail',
      'Narayanganj',        'Narsingdi',           'Natore',
        'Netrokona',       'Nilphamari',         'Noa

C:\Users\ASUS\AppData\Local\Temp\ipykernel_26448\4073995366.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  str_cols = df.select_dtypes(include=['object']).columns


In [12]:




# 2. Định nghĩa các ngưỡng thực tế cho các chỉ số quan trọng
# Các giá trị ngoài khoảng này sẽ được coi là bất hợp lý
valid_ranges = {
    'Avg Temp': (5, 50),      # Nhiệt độ trung bình từ 5°C đến 50°C
    'pH': (3.5, 9.5),         # Độ pH đất nông nghiệp thường từ 3.5 đến 9.5
    'Avg Humidity': (10, 100),# Độ ẩm không khí từ 10% đến 100%
    'Rainfall': (0, 5000),    # Lượng mưa không thể âm
    'LST_Kelvin': (270, 330), # Nhiệt độ bề mặt (tương đương -3°C đến 57°C)
    'Nitrogen': (0, 100),     # Hàm lượng đạm (%)
    'Clay': (0, 100),         # % thành phần đất
    'Sand': (0, 100),
    'Silt': (0, 100)
}

# 3. Hàm xử lý dữ liệu bất hợp lý
def fix_invalid_data(df, ranges):
    df_cleaned = df.copy()
    for col, (min_val, max_val) in ranges.items():
        if col in df_cleaned.columns:
            # Đếm số lượng lỗi trước khi sửa
            invalid_mask = (df_cleaned[col] < min_val) | (df_cleaned[col] > max_val)
            num_invalid = invalid_mask.sum()
            
            if num_invalid > 0:
                print(f"Cột {col}: Phát hiện {num_invalid} giá trị bất hợp lý. Đang xử lý...")
                # Thay thế giá trị sai bằng Median của cột đó để tránh làm lệch dữ liệu
                median_val = df_cleaned[col].median()
                df_cleaned.loc[invalid_mask, col] = median_val
                
    return df_cleaned

# Thực hiện xóa/sửa dữ liệu sai
df_cleaned = fix_invalid_data(df, valid_ranges)

# 4. Kiểm tra logic tổng thành phần đất (Clay + Sand + Silt phải xấp xỉ 100%)
# Nếu tổng sai lệch quá nhiều (> 5%), chúng ta điều chỉnh tỷ lệ
soil_parts = ['Clay', 'Sand', 'Silt']
if all(col in df_cleaned.columns for col in soil_parts):
    soil_sum = df_cleaned[soil_parts].sum(axis=1)
    # Tìm các dòng có tổng sai lệch lớn hơn 1%
    bad_soil_logic = (soil_sum < 99) | (soil_sum > 101)
    if bad_soil_logic.any():
        print(f"Phát hiện {bad_soil_logic.sum()} dòng sai logic tổng thành phần đất. Đang chuẩn hóa...")
        # Chuẩn hóa về tỷ lệ 100%
        df_cleaned.loc[bad_soil_logic, soil_parts] = df_cleaned.loc[bad_soil_logic, soil_parts].div(soil_sum[bad_soil_logic], axis=0) * 100

# 5. Lưu kết quả
print("\n--- Hoàn thành xử lý dữ liệu bất hợp lý ---")



--- Hoàn thành xử lý dữ liệu bất hợp lý ---


In [13]:

import re
import calendar

# ===============================
# COPY DATA
# ===============================
df = df.copy()

# ===============================
# 1. CHUẨN HÓA TOÀN BỘ STRING
# ===============================
str_cols = df.select_dtypes(include="object").columns

for col in str_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

# ===============================
# 2. CHUẨN HÓA TRANSPLANT (FULL MONTH NAME)
# ===============================
month_map = {
    'Jan': 'January', 'Feb': 'February',
    'Aug': 'August', 'Sep': 'September',
    'Oct': 'October', 'Nov': 'November',
    'Dec': 'December'
}

df['Transplant'] = df['Transplant'].replace(month_map)

# ===============================
# 3. FIX TYPO & CHUẨN HÓA TITLE CASE
# ===============================
cycle_cols = ['Transplant', 'Growth', 'Harvest']

df[cycle_cols] = df[cycle_cols].apply(lambda x: x.str.title())

df.replace({
    'Throuout The Year': 'Throughout The Year'
}, inplace=True)

# ===============================
# 4. IMPUTE CROP CYCLE THEO SEASON (VECTORIZE)
# ===============================
season_rules = {
    'Kharif 1': {
        'Transplant': 'March',
        'Growth': 'April To May',
        'Harvest': 'June'
    },
    'Kharif 2': {
        'Transplant': 'July',
        'Growth': 'August To October',
        'Harvest': 'November'
    },
    'Rabi': {
        'Transplant': 'November',
        'Growth': 'December To February',
        'Harvest': 'March'
    }
}

invalid_vals = ['No Need To Do', 'Throughout The Year']

for season, rules in season_rules.items():
    season_mask = df['Season'] == season
    
    for col in cycle_cols:
        invalid_mask = df[col].isin(invalid_vals)
        df.loc[season_mask & invalid_mask, col] = rules[col]

# ===============================
# 5. LOẠI BỎ TIẾNG VIỆT TRONG NGOẶC
# ===============================
df['pH_Suitability'] = df['pH_Suitability'].str.replace(r"\s*\(.*\)", "", regex=True)
df['Dominant_Soil_Texture'] = df['Dominant_Soil_Texture'].str.replace(r"\s*\(.*\)", "", regex=True)

# ===============================
# 6. DROP CỘT KHÔNG CÓ VARIANCE
# ===============================
low_variance_cols = [col for col in df.columns if df[col].nunique() <= 1]
df.drop(columns=low_variance_cols, inplace=True)

# ===============================
# 7. TÍNH THỜI GIAN SINH TRƯỞNG (THÁNG)
# ===============================
month_to_num = {month: index for index, month in enumerate(calendar.month_name) if month}

def duration_months(text):
    if isinstance(text, str) and "To" in text:
        parts = text.split("To")
        if len(parts) == 2:
            start = parts[0].strip()
            end = parts[1].strip()
            if start in month_to_num and end in month_to_num:
                return (month_to_num[end] - month_to_num[start]) % 12
    return np.nan

df['Growth_Duration_Months'] = df['Growth'].apply(duration_months)

# ===============================
# 8. VALIDATION
# ===============================
print("Shape after cleaning:", df.shape)
print("\nDropped columns:", low_variance_cols)
print("\nUnique Seasons:", df['Season'].unique())
print("\nUnique Transplant:", sorted(df['Transplant'].unique()))
print("\nSample Growth Duration:")
print(df[['Growth','Growth_Duration_Months']].head())

C:\Users\ASUS\AppData\Local\Temp\ipykernel_26448\4286739299.py:12: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  str_cols = df.select_dtypes(include="object").columns


Shape after cleaning: (4178, 51)

Dropped columns: ['Year', 'Compaction_Risk']

Unique Seasons: <StringArray>
['Rabi', 'Kharif 1', 'Kharif 2']
Length: 3, dtype: str

Unique Transplant: ['April', 'August', 'December', 'February', 'January', 'July', 'June', 'March', 'May', 'November', 'October', 'September']

Sample Growth Duration:
         Growth  Growth_Duration_Months
0  Jan To March                     NaN
1  Jan To March                     NaN
2  Dec To March                     NaN
3  Dec To March                     NaN
4  April To May                     1.0


In [14]:
str_cols = df.select_dtypes(include=['object']).columns

for col in str_cols:
    print(f"\nColumn: {col}")
    print(df[col].unique())


Column: District
<StringArray>
[        'Bagerhat',        'Bandarban',          'Barguna',
         'Barishal',            'Bhola',           'Bogura',
     'Brahmanbaria',         'Chandpur', 'Chapai Nawabganj',
       'Chattogram',        'Chuadanga',        'Coxsbazar',
          'Cumilla',            'Dhaka',         'Dinajpur',
         'Faridpur',             'Feni',        'Gaibandha',
          'Gazipur',        'Gopalganj',         'Habiganj',
         'Jamalpur',          'Jashore',       'Jhallokati',
        'Jhenaidah',        'Joypurhat',      'Khagrachari',
           'Khulna',      'Kishoreganj',         'Kurigram',
          'Kushtia',       'Lakshmipur',      'Lalmonirhat',
        'Madaripur',           'Magura',        'Manikganj',
         'Meherpur',      'Moulvibazar',       'Munshiganj',
       'Mymensingh',          'Naogaon',           'Narail',
      'Narayanganj',        'Narsingdi',           'Natore',
        'Netrokona',       'Nilphamari',         'Noa

C:\Users\ASUS\AppData\Local\Temp\ipykernel_26448\4073995366.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  str_cols = df.select_dtypes(include=['object']).columns


## Bước 5 : Xử lý những giá trị ngoại lai

1. Mục tiêu chiến dịchLoại bỏ nhiễu (noise) từ dữ liệu vệ tinh và thiết bị cảm biến, đồng thời bảo toàn tín hiệu thiên tai (Heat Stress, Wind Max) để không làm mất đi các yếu tố ảnh hưởng mạnh đến năng suất cây trồng.2. Các vấn đề đã khắc phụcTránh xử lý máy móc: Thay vì "cắt bằng" tất cả các cột, chúng ta đã phân loại biến theo đặc tính sinh học và khí tượng.Giảm sai lệch (Bias): Ngăn chặn việc mô hình bị chệch hướng bởi các giá trị quá lớn do lỗi thiết bị đo.Giữ dấu vết sự kiện: Đảm bảo các năm có thời tiết khắc nghiệt vẫn được mô hình ghi nhận thông qua cơ chế gắn thẻ (Flagging).3. Chiến lược xử lý chi tiếtA. Nhóm Chỉ số Vệ tinh & Đất (Capping)Danh sách: Nitrogen, Organic_Carbon, LAI, EVI, FPAR, AP Ratio, Humidity.Phương pháp: Sử dụng dải IQR (Interquartile Range).Công thức:$IQR = Q3 - Q1$$Lower\ Bound = Q1 - 1.5 \times IQR$$Upper\ Bound = Q3 + 1.5 \times IQR$Hành động: Thu gọn các giá trị nằm ngoài biên về giá trị biên gần nhất bằng np.clip.B. Nhóm Thời tiết Cực đoan (Flagging & Clipping)Danh sách: Heat_Stress_Days, Wind_Max.Hành động:Tạo cột đánh dấu: is_extreme_[Column] = 1 nếu là ngoại lai.Sau đó mới thực hiện thu gọn giá trị.Mục đích: Giúp mô hình học được rằng: "Đây là một năm có biến động lớn, cần chú ý đặc biệt đến năng suất".

In [15]:


def smart_agri_cleaner(df, capping_cols, extreme_event_cols):
    """
    Xử lý Outliers thông minh cho dữ liệu Nông nghiệp.
    - capping_cols: Các cột chỉ cần giới hạn biên (EVI, LAI, Dinh dưỡng đất).
    - extreme_event_cols: Các cột thời tiết cần đánh dấu thiên tai trước khi xử lý.
    """
    df_final = df.copy()
    
    # 1. Xử lý nhóm Biến Khí hậu (Cần đánh dấu trước khi Cap)
    for col in extreme_event_cols:
        if col in df_final.columns:
            Q1 = df_final[col].quantile(0.25)
            Q3 = df_final[col].quantile(0.75)
            IQR = Q3 - Q1
            upper_bound = Q3 + 1.5 * IQR
            
            # Tạo cột Flag: 1 nếu là thiên tai/cực đoan, 0 nếu bình thường
            df_final[f'is_extreme_{col}'] = (df_final[col] > upper_bound).astype(int)
            
            # Sau đó mới thực hiện Capping để ổn định phương sai cho mô hình
            df_final[col] = np.clip(df_final[col], None, upper_bound)
            print(f"[v] Đã Flag và Cap cột cực đoan: {col}")

    # 2. Xử lý nhóm Biến Chỉ số & Đất (Capping thông thường)
    for col in capping_cols:
        if col in df_final.columns:
            Q1 = df_final[col].quantile(0.25)
            Q3 = df_final[col].quantile(0.75)
            IQR = Q3 - Q1
            lower_bound = Q1 - 1.5 * IQR
            upper_bound = Q3 + 1.5 * IQR
            
            # Sử dụng np.clip để code gọn và nhanh hơn np.where
            df_final[col] = np.clip(df_final[col], lower_bound, upper_bound)
    
    print(f"[-] Hoàn thành xử lý {len(capping_cols) + len(extreme_event_cols)} cột.")
    return df_final

# --- ÁP DỤNG VÀO DỮ LIỆU CỦA BẠN ---

# Nhóm 1: Các biến chỉ số (Vệ tinh, Đất, Độ ẩm) - Dùng Capping
capping_list = [
     'Nitrogen', 'LAI', 'Organic_Carbon',
    'Max Relative Humidity', 'Min Relative Humidity', 'Avg Humidity', 'EVI', 'FPAR'
]

# Nhóm 2: Các biến có tính "Thiên tai/Cực đoan" - Dùng Flagging + Capping
extreme_list = ['Heat_Stress_Days', 'Wind_Max']

# Thực hiện xử lý
df = smart_agri_cleaner(df, capping_list, extreme_list)
df.to_csv('Agri_Data_Cleaned.csv', index=False, encoding='utf-8-sig')

[v] Đã Flag và Cap cột cực đoan: Heat_Stress_Days
[v] Đã Flag và Cap cột cực đoan: Wind_Max
[-] Hoàn thành xử lý 10 cột.


In [16]:
df.head()

,Area,AP Ratio,District,Season,Avg Temp,Avg Humidity,Crop Name,Transplant,Growth,Harvest,...,Heat_Stress_Days,Wind_Mean,Wind_Max,Wind_Min,Rain_Temp_Ratio,Extreme_Heat_Risk,Is_Extreme_Heat,Growth_Duration_Months,is_extreme_Heat_Stress_Days,is_extreme_Wind_Max
0,259,0.772201,Bagerhat,Rabi,17.75,55.0,Wheat,December,Jan To March,April,...,0.0,1.452917,4.33,0.04,11.23,Low Risk,0,NaN,0,0
1,421,2.247031,Bagerhat,Rabi,20.50,60.0,Maize 2,December,Jan To March,April,...,0.0,1.452917,4.33,0.04,11.23,Low Risk,0,NaN,0,0
2,145439,1.847881,Bagerhat,Rabi,26.00,72.5,Boro,November,Dec To March,April,...,0.0,1.452917,4.33,0.04,11.23,Low Risk,0,NaN,0,0
3,627,3.508772,Bagerhat,Rabi,20.50,80.0,Sweet Potato,November,Dec To March,April,...,0.0,1.452917,4.33,0.04,11.23,Low Risk,0,NaN,0,0
4,383,4.994778,Bagerhat,Kharif 1,31.50,70.0,Mango,April,April To May,April To June,...,42.5,2.481885,5.86,0.04,22.73,High Risk,1,1.0,1,0


In [17]:
df.info()

<class 'pandas.DataFrame'>
Index: 4178 entries, 0 to 4607
Data columns (total 53 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Area                         4178 non-null   int64  
 1   AP Ratio                     4178 non-null   float64
 2   District                     4178 non-null   str    
 3   Season                       4178 non-null   str    
 4   Avg Temp                     4178 non-null   float64
 5   Avg Humidity                 4178 non-null   float64
 6   Crop Name                    4178 non-null   str    
 7   Transplant                   4178 non-null   str    
 8   Growth                       4178 non-null   str    
 9   Harvest                      4178 non-null   str    
 10  Production                   4178 non-null   int64  
 11  Max Temp                     4178 non-null   int64  
 12  Min Temp                     4178 non-null   float64
 13  Max Relative Humidity        4178 

In [18]:
str_cols = df.select_dtypes(include=['object']).columns

for col in str_cols:
    print(f"\nColumn: {col}")
    print(df[col].unique())


Column: District
<StringArray>
[        'Bagerhat',        'Bandarban',          'Barguna',
         'Barishal',            'Bhola',           'Bogura',
     'Brahmanbaria',         'Chandpur', 'Chapai Nawabganj',
       'Chattogram',        'Chuadanga',        'Coxsbazar',
          'Cumilla',            'Dhaka',         'Dinajpur',
         'Faridpur',             'Feni',        'Gaibandha',
          'Gazipur',        'Gopalganj',         'Habiganj',
         'Jamalpur',          'Jashore',       'Jhallokati',
        'Jhenaidah',        'Joypurhat',      'Khagrachari',
           'Khulna',      'Kishoreganj',         'Kurigram',
          'Kushtia',       'Lakshmipur',      'Lalmonirhat',
        'Madaripur',           'Magura',        'Manikganj',
         'Meherpur',      'Moulvibazar',       'Munshiganj',
       'Mymensingh',          'Naogaon',           'Narail',
      'Narayanganj',        'Narsingdi',           'Natore',
        'Netrokona',       'Nilphamari',         'Noa

C:\Users\ASUS\AppData\Local\Temp\ipykernel_26448\4073995366.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  str_cols = df.select_dtypes(include=['object']).columns


#### tạo ra cột Yeild và xóa bỏ drop bỏ những cột Growth   

In [19]:
df = df.drop('Growth_Duration_Months', axis=1)
df['Yield'] = df['Production'] / df['Area']
df.to_csv('Agri_Data_Cleaned.csv', index=False, encoding='utf-8-sig')

In [20]:
df.info()

<class 'pandas.DataFrame'>
Index: 4178 entries, 0 to 4607
Data columns (total 53 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Area                         4178 non-null   int64  
 1   AP Ratio                     4178 non-null   float64
 2   District                     4178 non-null   str    
 3   Season                       4178 non-null   str    
 4   Avg Temp                     4178 non-null   float64
 5   Avg Humidity                 4178 non-null   float64
 6   Crop Name                    4178 non-null   str    
 7   Transplant                   4178 non-null   str    
 8   Growth                       4178 non-null   str    
 9   Harvest                      4178 non-null   str    
 10  Production                   4178 non-null   int64  
 11  Max Temp                     4178 non-null   int64  
 12  Min Temp                     4178 non-null   float64
 13  Max Relative Humidity        4178 